In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
plant1 = pd.read_csv("../data/final/plant1_features.csv")
plant2 = pd.read_csv("../data/final/plant2_features.csv")

plant1["DATE_TIME"] = pd.to_datetime(plant1["DATE_TIME"])
plant2["DATE_TIME"] = pd.to_datetime(plant2["DATE_TIME"])

print("Plant 1 shape:", plant1.shape)
print("Plant 2 shape:", plant2.shape)

Plant 1 shape: (66640, 25)
Plant 2 shape: (65564, 25)


In [3]:
feature_columns = [
    "hour",
    "minute",
    "day",
    "day_of_week",
    "day_of_year",
    "month",
    "hour_sin",
    "hour_cos",
    "AC_POWER_lag_1",
    "AC_POWER_lag_4",
    "AC_POWER_lag_8",
    "AC_POWER_lag_96",
    "AC_POWER_rolling_mean_4",
    "AC_POWER_rolling_mean_8"
]

target_column = "TARGET_AC_POWER"

print("Number of features:", len(feature_columns))
print("Target:", target_column)

Number of features: 14
Target: TARGET_AC_POWER


In [12]:
def chronological_split(df, train_ratio=0.70, val_ratio=0.15):

    df = df.sort_values("DATE_TIME").copy()

    unique_times = np.sort(df["DATE_TIME"].unique())

    n_times = len(unique_times)

    train_end = int(n_times * train_ratio)
    val_end = int(n_times * (train_ratio + val_ratio))

    train_end_time = unique_times[train_end]
    val_end_time = unique_times[val_end]

    train = df[df["DATE_TIME"] < train_end_time].copy()

    validation = df[
        (df["DATE_TIME"] >= train_end_time) &
        (df["DATE_TIME"] < val_end_time)
    ].copy()

    test = df[df["DATE_TIME"] >= val_end_time].copy()

    return train, validation, test

In [13]:
plant1_train, plant1_val, plant1_test = chronological_split(plant1)

plant2_train, plant2_val, plant2_test = chronological_split(plant2)

print("Plant 1:")
print("Train:", plant1_train.shape)
print("Validation:", plant1_val.shape)
print("Test:", plant1_test.shape)

print("\nPlant 2:")
print("Train:", plant2_train.shape)
print("Validation:", plant2_val.shape)
print("Test:", plant2_test.shape)

Plant 1:
Train: (46444, 25)
Validation: (10098, 25)
Test: (10098, 25)

Plant 2:
Train: (44686, 25)
Validation: (10428, 25)
Test: (10450, 25)


In [14]:
X1_train = plant1_train[feature_columns]
y1_train = plant1_train[target_column]

X1_val = plant1_val[feature_columns]
y1_val = plant1_val[target_column]

X1_test = plant1_test[feature_columns]
y1_test = plant1_test[target_column]


X2_train = plant2_train[feature_columns]
y2_train = plant2_train[target_column]

X2_val = plant2_val[feature_columns]
y2_val = plant2_val[target_column]

X2_test = plant2_test[feature_columns]
y2_test = plant2_test[target_column]

In [15]:
print("Plant 1:")
print("X_train:", X1_train.shape)
print("y_train:", y1_train.shape)
print("X_val:", X1_val.shape)
print("X_test:", X1_test.shape)

print("\nPlant 2:")
print("X_train:", X2_train.shape)
print("y_train:", y2_train.shape)
print("X_val:", X2_val.shape)
print("X_test:", X2_test.shape)

Plant 1:
X_train: (46444, 14)
y_train: (46444,)
X_val: (10098, 14)
X_test: (10098, 14)

Plant 2:
X_train: (44686, 14)
y_train: (44686,)
X_val: (10428, 14)
X_test: (10450, 14)


In [16]:
# What this cell does:
# Train a Linear Regression model using only the training data.
# The model learns the relationship between our 14 engineered features
# and the 15-minute-ahead target: TARGET_AC_POWER.

linear_model_plant1 = LinearRegression()
linear_model_plant2 = LinearRegression()

linear_model_plant1.fit(X1_train, y1_train)
linear_model_plant2.fit(X2_train, y2_train)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


In [18]:
# What this cell does:
# Generate predictions for the validation data using
# the trained Linear Regression models.
#
# The models were trained on the training data only.
# Now we use them to predict the unseen validation period.

y1_val_pred = linear_model_plant1.predict(X1_val)
y2_val_pred = linear_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_val_pred.shape)
print("Plant 2 validation predictions:", y2_val_pred.shape)

Plant 1 validation predictions: (10098,)
Plant 2 validation predictions: (10428,)


In [19]:
# What this cell does:
# Evaluate Linear Regression on the validation data using:
# MAE  -> average absolute prediction error
# RMSE -> gives more penalty to large errors
# R²   -> proportion of target variation explained by the model

lr_plant1_val_mae = mean_absolute_error(y1_val, y1_val_pred)
lr_plant1_val_rmse = np.sqrt(mean_squared_error(y1_val, y1_val_pred))
lr_plant1_val_r2 = r2_score(y1_val, y1_val_pred)

lr_plant2_val_mae = mean_absolute_error(y2_val, y2_val_pred)
lr_plant2_val_rmse = np.sqrt(mean_squared_error(y2_val, y2_val_pred))
lr_plant2_val_r2 = r2_score(y2_val, y2_val_pred)

print("Plant 1 - Linear Regression Validation")
print("MAE :", lr_plant1_val_mae)
print("RMSE:", lr_plant1_val_rmse)
print("R²  :", lr_plant1_val_r2)

print("\nPlant 2 - Linear Regression Validation")
print("MAE :", lr_plant2_val_mae)
print("RMSE:", lr_plant2_val_rmse)
print("R²  :", lr_plant2_val_r2)

Plant 1 - Linear Regression Validation
MAE : 76.13534189180926
RMSE: 112.47622114070722
R²  : 0.9073729162277532

Plant 2 - Linear Regression Validation
MAE : 84.18313166700969
RMSE: 153.9074423866472
R²  : 0.7652164495255714


In [20]:
# What this cell does:
# Create Persistence predictions on the validation period.
# Persistence predicts the next 15-minute AC power using
# the current AC_POWER value.

y1_val_persistence = plant1_val["AC_POWER"]
y2_val_persistence = plant2_val["AC_POWER"]

print("Plant 1 persistence validation predictions:", y1_val_persistence.shape)
print("Plant 2 persistence validation predictions:", y2_val_persistence.shape)

Plant 1 persistence validation predictions: (10098,)
Plant 2 persistence validation predictions: (10428,)


In [21]:
# What this cell does:
# Calculate MAE, RMSE and R² for the Persistence baseline
# on exactly the same validation period used by Linear Regression.

persistence_plant1_val_mae = mean_absolute_error(
    y1_val, y1_val_persistence
)

persistence_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_val_persistence)
)

persistence_plant1_val_r2 = r2_score(
    y1_val, y1_val_persistence
)


persistence_plant2_val_mae = mean_absolute_error(
    y2_val, y2_val_persistence
)

persistence_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_val_persistence)
)

persistence_plant2_val_r2 = r2_score(
    y2_val, y2_val_persistence
)


print("Plant 1 - Persistence Validation")
print("MAE :", persistence_plant1_val_mae)
print("RMSE:", persistence_plant1_val_rmse)
print("R²  :", persistence_plant1_val_r2)

print("\nPlant 2 - Persistence Validation")
print("MAE :", persistence_plant2_val_mae)
print("RMSE:", persistence_plant2_val_rmse)
print("R²  :", persistence_plant2_val_r2)

Plant 1 - Persistence Validation
MAE : 51.873702536856804
RMSE: 100.6603795499464
R²  : 0.9258119963809976

Plant 2 - Persistence Validation
MAE : 55.474724972636366
RMSE: 121.91543195823563
R²  : 0.8526786170717655


In [22]:
# What this cell does:
# Import the Random Forest regression algorithm.
# Random Forest combines multiple decision trees to learn
# nonlinear relationships between features and AC power.

from sklearn.ensemble import RandomForestRegressor

In [23]:
# What this cell does:
# Train separate Random Forest regression models for Plant 1 and Plant 2.
#
# n_estimators = 200:
# Build 200 decision trees.
#
# random_state = 42:
# Makes the experiment reproducible.
#
# n_jobs = -1:
# Use all available CPU cores.

rf_model_plant1 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_plant2 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_plant1.fit(X1_train, y1_train)
rf_model_plant2.fit(X2_train, y2_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [24]:
# What this cell does:
# Use the trained Random Forest models to predict
# the unseen validation period.

y1_rf_val_pred = rf_model_plant1.predict(X1_val)
y2_rf_val_pred = rf_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_rf_val_pred.shape)
print("Plant 2 validation predictions:", y2_rf_val_pred.shape)

Plant 1 validation predictions: (10098,)
Plant 2 validation predictions: (10428,)


In [25]:
# What this cell does:
# Evaluate Random Forest using MAE, RMSE and R²
# on the validation period.

rf_plant1_val_mae = mean_absolute_error(
    y1_val, y1_rf_val_pred
)

rf_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_rf_val_pred)
)

rf_plant1_val_r2 = r2_score(
    y1_val, y1_rf_val_pred
)


rf_plant2_val_mae = mean_absolute_error(
    y2_val, y2_rf_val_pred
)

rf_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_rf_val_pred)
)

rf_plant2_val_r2 = r2_score(
    y2_val, y2_rf_val_pred
)


print("Plant 1 - Random Forest Validation")
print("MAE :", rf_plant1_val_mae)
print("RMSE:", rf_plant1_val_rmse)
print("R²  :", rf_plant1_val_r2)

print("\nPlant 2 - Random Forest Validation")
print("MAE :", rf_plant2_val_mae)
print("RMSE:", rf_plant2_val_rmse)
print("R²  :", rf_plant2_val_r2)

Plant 1 - Random Forest Validation
MAE : 71.44842941759312
RMSE: 143.02773472329633
R²  : 0.8502188785607956

Plant 2 - Random Forest Validation
MAE : 84.32230769847156
RMSE: 171.8142012595407
R²  : 0.7074052438683532


In [26]:
# What this cell does:
# Import XGBRegressor, the gradient-boosting regression algorithm
# we will use as our main classical ML forecasting model.

from xgboost import XGBRegressor

print("XGBoost imported successfully.")

XGBoost imported successfully.


In [27]:
# What this cell does:
# Create separate XGBoost regression models for Plant 1 and Plant 2.
#
# n_estimators = number of boosting trees
# learning_rate = contribution of each tree
# max_depth = maximum depth of each tree
# random_state = reproducible results
#
# These are initial parameters, not final hyperparameter tuning.

xgb_model_plant1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_model_plant2 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

print("XGBoost models created successfully.")

XGBoost models created successfully.


In [28]:
# What this cell does:
# Train XGBoost using only the training data.
#
# The validation data remains unseen during training.

xgb_model_plant1.fit(X1_train, y1_train)
xgb_model_plant2.fit(X2_train, y2_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [29]:
# What this cell does:
# Generate predictions for the unseen validation period
# using the trained XGBoost models.

y1_xgb_val_pred = xgb_model_plant1.predict(X1_val)
y2_xgb_val_pred = xgb_model_plant2.predict(X2_val)

print("Plant 1 validation predictions:", y1_xgb_val_pred.shape)
print("Plant 2 validation predictions:", y2_xgb_val_pred.shape)

Plant 1 validation predictions: (10098,)
Plant 2 validation predictions: (10428,)


In [30]:
# What this cell does:
# Calculate MAE, RMSE and R² for XGBoost
# on the validation period.

xgb_plant1_val_mae = mean_absolute_error(
    y1_val, y1_xgb_val_pred
)

xgb_plant1_val_rmse = np.sqrt(
    mean_squared_error(y1_val, y1_xgb_val_pred)
)

xgb_plant1_val_r2 = r2_score(
    y1_val, y1_xgb_val_pred
)


xgb_plant2_val_mae = mean_absolute_error(
    y2_val, y2_xgb_val_pred
)

xgb_plant2_val_rmse = np.sqrt(
    mean_squared_error(y2_val, y2_xgb_val_pred)
)

xgb_plant2_val_r2 = r2_score(
    y2_val, y2_xgb_val_pred
)


print("Plant 1 - XGBoost Validation")
print("MAE :", xgb_plant1_val_mae)
print("RMSE:", xgb_plant1_val_rmse)
print("R²  :", xgb_plant1_val_r2)

print("\nPlant 2 - XGBoost Validation")
print("MAE :", xgb_plant2_val_mae)
print("RMSE:", xgb_plant2_val_rmse)
print("R²  :", xgb_plant2_val_r2)

Plant 1 - XGBoost Validation
MAE : 66.30495723649975
RMSE: 132.37782218227156
R²  : 0.8716939836320359

Plant 2 - XGBoost Validation
MAE : 80.38963778971036
RMSE: 168.26032009383576
R²  : 0.7193843778070619


In [31]:
# What this cell does:
# Combine the validation results of all four forecasting approaches
# into one table so we can compare them using the same dataset split
# and the same evaluation metrics.

validation_results = pd.DataFrame({
    "Plant": [
        "Plant 1", "Plant 1", "Plant 1", "Plant 1",
        "Plant 2", "Plant 2", "Plant 2", "Plant 2"
    ],

    "Model": [
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],

    "MAE": [
        persistence_plant1_val_mae,
        lr_plant1_val_mae,
        rf_plant1_val_mae,
        xgb_plant1_val_mae,
        persistence_plant2_val_mae,
        lr_plant2_val_mae,
        rf_plant2_val_mae,
        xgb_plant2_val_mae
    ],

    "RMSE": [
        persistence_plant1_val_rmse,
        lr_plant1_val_rmse,
        rf_plant1_val_rmse,
        xgb_plant1_val_rmse,
        persistence_plant2_val_rmse,
        lr_plant2_val_rmse,
        rf_plant2_val_rmse,
        xgb_plant2_val_rmse
    ],

    "R2": [
        persistence_plant1_val_r2,
        lr_plant1_val_r2,
        rf_plant1_val_r2,
        xgb_plant1_val_r2,
        persistence_plant2_val_r2,
        lr_plant2_val_r2,
        rf_plant2_val_r2,
        xgb_plant2_val_r2
    ]
})

validation_results

,Plant,Model,MAE,RMSE,R2
0,Plant 1,Persistence,51.873703,100.660380,0.925812
1,Plant 1,Linear Regression,76.135342,112.476221,0.907373
2,Plant 1,Random Forest,71.448429,143.027735,0.850219
3,Plant 1,XGBoost,66.304957,132.377822,0.871694
4,Plant 2,Persistence,55.474725,121.915432,0.852679
5,Plant 2,Linear Regression,84.183132,153.907442,0.765216
6,Plant 2,Random Forest,84.322308,171.814201,0.707405
7,Plant 2,XGBoost,80.389638,168.260320,0.719384


In [33]:
# What this cell does:
# Save the complete validation model comparison to the
# outputs/metrics/ folder for later analysis.

import os

METRICS_PATH = "../outputs/metrics/"
os.makedirs(METRICS_PATH, exist_ok=True)

validation_results.to_csv(
    METRICS_PATH + "validation_model_results.csv",
    index=False
)

print("Validation model comparison saved successfully.")

Validation model comparison saved successfully.


In [34]:
# What this cell does:
# Save the trained Linear Regression, Random Forest,
# and XGBoost models so they can be reused later
# without retraining.

import joblib
import os

MODEL_PATH = "../models/"

os.makedirs(MODEL_PATH + "linear_regression/", exist_ok=True)
os.makedirs(MODEL_PATH + "random_forest/", exist_ok=True)
os.makedirs(MODEL_PATH + "xgboost/", exist_ok=True)

# Linear Regression
joblib.dump(
    linear_model_plant1,
    MODEL_PATH + "linear_regression/plant1_model.pkl"
)

joblib.dump(
    linear_model_plant2,
    MODEL_PATH + "linear_regression/plant2_model.pkl"
)

# Random Forest
joblib.dump(
    rf_model_plant1,
    MODEL_PATH + "random_forest/plant1_model.pkl"
)

joblib.dump(
    rf_model_plant2,
    MODEL_PATH + "random_forest/plant2_model.pkl"
)

# XGBoost
joblib.dump(
    xgb_model_plant1,
    MODEL_PATH + "xgboost/plant1_model.pkl"
)

joblib.dump(
    xgb_model_plant2,
    MODEL_PATH + "xgboost/plant2_model.pkl"
)

print("All trained models saved successfully.")

All trained models saved successfully.
